In [1]:
import numpy as np
import random
import pandas as pd

# Import các lớp nhiệm vụ từ dự án
from candy_party import CandyParty
from flu_vaccine import FluVaccine
from flower_garden import FlowerGarden
from clinical_notes import ClinicalNotes
from cell_bio import CellBio
from utils import Utils

In [2]:
from task_generator import TaskGenerator

def get_root_fixed(self, dag, return_name=True):
    roots = [v for v, d in dag.in_degree() if d == 0]
    if len(roots) > 0:
        return roots[0]
    return list(dag.nodes())[0]

def get_leaf_fixed(self, dag):
    leaves = [v for v, d in dag.out_degree() if d == 0]
    if len(leaves) > 0:
        return leaves[0]
    return list(dag.nodes())[-1]

TaskGenerator.get_root = get_root_fixed
TaskGenerator.get_leaf = get_leaf_fixed


from candy_party import CandyParty
from flu_vaccine import FluVaccine
from flower_garden import FlowerGarden
from clinical_notes import ClinicalNotes
from cell_bio import CellBio
from utils import Utils

In [3]:
np.random.seed(0)
random.seed(0)
utils = Utils()

print("Initializing CandyParty model...")
cp = CandyParty(
    n_per_bcc=[4, 3, 3], 
    bcc_types=['cycle', 'cycle', 'cycle'], 
    causal_functions='or', 
    plot=False
)

X = cp.root
Y = cp.leaf
C = cp.cutpoints[0]
D = cp.cutpoints[1]
print(f"Graph structure: Root X={X} -> Cutpoint C={C} -> Cutpoint D={D} -> Leaf Y={Y}\n")

Initializing CandyParty model...
Graph structure: Root X=Ilma -> Cutpoint C=Lorraine -> Cutpoint D=Darla -> Leaf Y=Marcia



In [4]:
print("==================================================")
print("COMPUTING GROUND TRUTH PNS VIA SCM (N=100,000)")
print("==================================================")

pns_pairs = {
    "PNS_XY (Global)": (X, Y),
    "PNS_XC (Local)": (X, C),
    "PNS_XD (Local)": (X, D),
    "PNS_CD (Local)": (C, D),
    "PNS_CY (Local)": (C, Y),
    "PNS_DY (Local)": (D, Y)
}
pns_results = {}

for label, (cause, effect) in pns_pairs.items():
    df1, _ = cp.sample_scm(n=100000, intervene_node=cause, intervene_value=1, seed=0)
    df0, _ = cp.sample_scm(n=100000, intervene_node=cause, intervene_value=0, seed=0)
    
    pns_val = np.mean(df1[effect].values & ~df0[effect].values)
    pns_results[label] = pns_val
    print(f"Calculated {label:<16}: {pns_val:.6f}")

COMPUTING GROUND TRUTH PNS VIA SCM (N=100,000)
Calculated PNS_XY (Global) : 0.000390
Calculated PNS_XC (Local)  : 0.048060
Calculated PNS_XD (Local)  : 0.005730
Calculated PNS_CD (Local)  : 0.121130
Calculated PNS_CY (Local)  : 0.009670
Calculated PNS_DY (Local)  : 0.081460


In [5]:
print("==================================================")
print("VERIFYING THEOREM 5.1 & MONTE CARLO ERROR")
print("==================================================")

global_pns = pns_results["PNS_XY (Global)"]
comp_1 = pns_results["PNS_XC (Local)"] * pns_results["PNS_CY (Local)"]
comp_2 = pns_results["PNS_XD (Local)"] * pns_results["PNS_DY (Local)"]
comp_3 = pns_results["PNS_XC (Local)"] * pns_results["PNS_CD (Local)"] * pns_results["PNS_DY (Local)"]
rae_1 = utils.get_rae(global_pns, comp_1)
rae_2 = utils.get_rae(global_pns, comp_2)
rae_3 = utils.get_rae(global_pns, comp_3)
print(f"Global PNS_XY (True Value): {global_pns:.6f}")
print(f"Composition 1 (PNS_XC * PNS_CY): {comp_1:.6f} | RAE: {rae_1*100:.2f}%")
print(f"Composition 2 (PNS_XD * PNS_DY): {comp_2:.6f} | RAE: {rae_2*100:.2f}%")
print(f"Composition 3 (PNS_XC * PNS_CD * PNS_DY): {comp_3:.6f} | RAE: {rae_3*100:.2f}%")

VERIFYING THEOREM 5.1 & MONTE CARLO ERROR
Global PNS_XY (True Value): 0.000390
Composition 1 (PNS_XC * PNS_CY): 0.000465 | RAE: 19.16%
Composition 2 (PNS_XD * PNS_DY): 0.000467 | RAE: 19.68%
Composition 3 (PNS_XC * PNS_CD * PNS_DY): 0.000474 | RAE: 21.59%


In [6]:
print("==================================================")
print("CROSS-THEME STRUCTURAL CONSISTENCY TEST")
print("==================================================")
fv = FluVaccine(n_per_bcc=[4, 3, 3], bcc_types=['cycle', 'cycle', 'cycle'], causal_functions='or', plot=False)
fg = FlowerGarden(n_per_bcc=[4, 3, 3], bcc_types=['cycle', 'cycle', 'cycle'], causal_functions='or', plot=False)
dag_match_fv = np.array_equal(cp.adj_dag, fv.adj_dag)
dag_match_fg = np.array_equal(cp.adj_dag, fg.adj_dag)
print(f"Does FluVaccine DAG match CandyParty DAG structurally?  : {dag_match_fv}")
print(f"Does FlowerGarden DAG match CandyParty DAG structurally?: {dag_match_fg}")
cn = ClinicalNotes(n_per_bcc=[4, 3, 3], bcc_types=['cycle', 'cycle', 'cycle'], causal_functions='or', plot=False)
print(f"\nClinicalNotes causal functions (non-leaves vs leaf):")
print(f"  - Non-leaf functions example: {cn.causal_functions[0]}")
print(f"  - Leaf function (last node) : {cn.causal_functions[-1]}")
print("\nInitializing CellBio model...")
cb = CellBio(
    n_per_bcc=[[2, 2, 2], [2, 2, 2]], 
    bcc_types=[['cycle', 'cycle', 'cycle'], ['cycle', 'cycle', 'cycle']], 
    causal_functions='or', 
    plot=False
)
df_cb, _ = cb.sample_scm(n=5, seed=0)
print("\nCellBio Sample SCM output (Continuous/Gaussian variables):")
print(df_cb.head(3))

CROSS-THEME STRUCTURAL CONSISTENCY TEST
Does FluVaccine DAG match CandyParty DAG structurally?  : True
Does FlowerGarden DAG match CandyParty DAG structurally?: True

ClinicalNotes causal functions (non-leaves vs leaf):
  - Non-leaf functions example: or
  - Leaf function (last node) : and

Initializing CellBio model...

CellBio Sample SCM output (Continuous/Gaussian variables):
   2STS  6C25  2ESE  ZX3S  XVZX  T08L
0     1     1     1     1     1     1
1     1     1     1     1     1     1
2     1     1     1     1     1     1
